Install dependencies

In [2]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.4 MB/s eta 0:00:00


Imports

In [3]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

Load TinyLlama (4-bit)

In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Stability tweaks
model.config.use_cache = False
model.gradient_checkpointing_enable()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Load your dataset

In [5]:
dataset = load_dataset("csv", data_files="final_pidgin_training_dataset.csv")
dataset = dataset["train"]

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 9884
})


Clean + format dataset

In [6]:
def format_example(example):
    return {"text": example["text"].strip()}

dataset = dataset.map(format_example)

Map:   0%|          | 0/9884 [00:00<?, ? examples/s]

Tokenize

In [7]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512   # safe for TinyLlama
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

Map:   0%|          | 0/9884 [00:00<?, ? examples/s]

LoRA config

In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

Apply LoRA

In [9]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


Training arguments

Trainer

In [10]:
training_args = TrainingArguments(
    output_dir="./tinyllama-pidgin",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1.5,
    learning_rate=2e-4,

    fp16=False,   # Changed to False
    bf16=False,   # Keep as False (or ensure it's False)

    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    report_to="none"
)

In [11]:
model = model.to(torch.float16)

In [12]:
def formatting_func(example):
    return f"### Instruction:\n{example['text']}\n\n### Response:\n"

In [13]:

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=formatting_func,
    args=training_args
)

Applying formatting function to train dataset:   0%|          | 0/9884 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/9884 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9884 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2767 > 2048). Running this sequence through the model will result in indexing errors


Train

In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,3.479790
100,2.899305
150,2.921135
200,2.921789
250,2.859637
300,2.822040
350,2.902446
400,2.725703
450,2.684273
500,2.723259


TrainOutput(global_step=3707, training_loss=2.592937261613836, metrics={'train_runtime': 6179.6957, 'train_samples_per_second': 2.399, 'train_steps_per_second': 0.6, 'total_flos': 4135619906002944.0, 'train_loss': 2.592937261613836})

save

In [ ]:
trainer.model.save_pretrained("./tinyllama-pidgin")
tokenizer.save_pretrained("./tinyllama-pidgin")

('./tinyllama-pidgin/tokenizer_config.json',
 './tinyllama-pidgin/chat_template.jinja',
 './tinyllama-pidgin/tokenizer.json')

In [15]:
prompt = "### Instruction:\nExplain why you are going to the market in pidgin\n\n### Response:\n"
prompt = "I dey go market because"

model.eval()
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

I dey go market because i dey feel like my boyfriend is not really my boyfriend

### Response:



In [16]:
for i in range(5):
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.6,
        top_p=0.85,
        top_k=40,
        repetition_penalty=1.5,
        no_repeat_ngram_size=4,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    print("\nResult:", i+1)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Result: 1
I dey go market because of this
<|assistant>

Result: 2
I dey go market because of my family i don forget
we no get money to buy food and we have a big house but the one thing that is important for us now e be like say na only me wan do this business oooo im tired thank god if

Result: 3
I dey go market because of this thing
awon jarele ohh e don tear me oooo i just want to sleep and eat something sweet as una say make dem give us amor for house like that we will never forget you all my love from

Result: 4
I dey go market because of this thing
i wan buy am but na my friend weh i get no money for shop oooo so e be like say the things come to you una know how much your friends and family members are willing or not going to give

Result: 5
I dey go market because of this
e be like say na only me and my family don buy food oo i no fit carry anything for house again now



In [18]:
import re

def clean_output(text):
    # remove repeated words
    text = re.sub(r'\b(\w+)( \1\b)+', r'\1', text)

    # remove noisy tokens (custom filter)
    bad_words = ["wet", "pip", "gu"]
    words = text.split()
    words = [w for w in words if w not in bad_words]

    return " ".join(words)

prompt = "I dey go market because"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

best = ""

for _ in range(10):
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.5,
        top_p=0.8,
        repetition_penalty=1.5,
        no_repeat_ngram_size=4,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    cleaned = clean_output(text)

    print("\nRaw:", text)
    print("Cleaned:", cleaned)

    if len(cleaned) > len(best):
        best = cleaned

print("\n✅ Best Output:\n", best)


Raw: I dey go market because i no fit buy food again
i don use am for my own life now oo e be like say dem wan kill me and sell the body as well bc of covid-1
Cleaned: I dey go market because i no fit buy food again i don use am for my own life now oo e be like say dem wan kill me and sell the body as well bc of covid-1

Raw: I dey go market because of this one thing
i don vex for dis life now i no fit sell am again if na me say we still do business like that
<|assistant>
Cleaned: I dey go market because of this one thing i don vex for dis life now i no fit sell am again if na me say we still do business like that <|assistant>

Raw: I dey go market because of my work i don chop am for months
make una give me money to buy food and clothes so that no one will know where im from naira or dollar e be like say dem
Cleaned: I dey go market because of my work i don chop am for months make una give me money to buy food and clothes so that no one will know where im from naira or dollar e be li

In [ ]:
prompt = "I dey go market because"

result = safe_generate(prompt)

print("Generated Output:\n")
print(result)

Generated Output:

I dey go market because I wan buy food for house.


In [ ]:
pip install streamlit transformers peft bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.5 MB/s eta 0:00:00


Create app.py

In [ ]:
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ----------------------------
# Load model (cached)
# ----------------------------
@st.cache_resource
def load_model():
    base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    adapter_path = "./tinyllama-pidgin"  # your saved model

    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto",
        dtype=torch.float16
    )

    model = PeftModel.from_pretrained(base_model, adapter_path)

    model.eval()
    return model, tokenizer


model, tokenizer = load_model()

# ----------------------------
# Streamlit UI
# ----------------------------
st.set_page_config(page_title="Pidgin Autocomplete", layout="centered")

st.title("🧠 Pidgin Autocomplete AI")
st.write("Type a sentence and get smart Pidgin completion")

# Input box
prompt = st.text_area("Enter your text:", height=150)

# Controls
max_tokens = st.slider("Max tokens", 10, 100, 50)
temperature = st.slider("Temperature", 0.1, 1.5, 0.7)

# ----------------------------
# Generate button
# ----------------------------
if st.button("Generate"):
    if prompt.strip() == "":
        st.warning("Please enter some text")
    else:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                top_p=0.9,
                do_sample=True
            )

        result = tokenizer.decode(outputs[0], skip_special_tokens=True)

        st.subheader("Completion:")
        st.write(result)

2026-04-11 15:51:43.202 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-04-11 15:52:01.267 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:52:01.268 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:52:01.345 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-11 15:52:01.346 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:52:01.347 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:52:01.348 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:52:01.349 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [ ]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [ ]:
!pip install pyngrok

Streaming output (real-time typing)

In [ ]:
import time

if st.button("Generate"):
    if prompt.strip() == "":
        st.warning("Please enter some text")
    else:
        placeholder = st.empty()

        result = safe_generate(prompt)

        streamed_text = ""

        for word in result.split():
            streamed_text += word + " "
            placeholder.markdown(streamed_text)
            time.sleep(0.05)  # 👈 makes it look like real typing

2026-04-11 15:57:29.447 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:57:29.452 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:57:29.455 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:57:29.457 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:57:29.459 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 15:57:29.461 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
